# Evaluation & Analysis — Sarcasm Style Transfer

Compare T5, GPT-2, and BART on sarcastic → non-sarcastic transfer.

**Sections**:
1. Load evaluation results
2. Overall metric comparison
3. Per-strategy breakdown
4. Sample output inspection
5. Human evaluation template

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

OUTPUTS_DIR = Path("../outputs")

# Models to compare — adjust paths after training
MODELS = {
    "T5-base": OUTPUTS_DIR / "t5-base" / "sar-to-non",
    "GPT-2": OUTPUTS_DIR / "gpt2" / "sar-to-non",
    "BART-base": OUTPUTS_DIR / "bart-base" / "sar-to-non",
}

## 1. Load Evaluation Results

In [ ]:
results = {}
samples = {}

for name, path in MODELS.items():
    eval_file = path / "eval_results.json"
    if eval_file.exists():
        with open(eval_file) as f:
            results[name] = json.load(f)
        print(f"Loaded {name}")
    else:
        print(f"MISSING: {eval_file}")

    sample_file = path / "sample_outputs.json"
    if sample_file.exists():
        with open(sample_file) as f:
            samples[name] = json.load(f)

## 2. Overall Metric Comparison

In [ ]:
rows = []
for name, r in results.items():
    m = r["metrics"]
    rows.append({"Model": name, "BLEU": m["bleu"], "METEOR": m["meteor"], "ROUGE-L": m["rouge_l"]})

df_metrics = pd.DataFrame(rows).set_index("Model")
display(df_metrics)

ax = df_metrics.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Sarcastic → Non-Sarcastic")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 3. Per-Strategy Breakdown

In [ ]:
strategy_rows = []
for name, r in results.items():
    for strategy, m in r.get("metrics_by_strategy", {}).items():
        strategy_rows.append({"Model": name, "Strategy": strategy, "BLEU": m["bleu"], "Count": m["count"]})

if strategy_rows:
    df_strat = pd.DataFrame(strategy_rows)
    display(df_strat.pivot(index="Strategy", columns="Model", values="BLEU"))

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=df_strat, x="Strategy", y="BLEU", hue="Model", ax=ax)
    ax.set_title("BLEU by Sarcasm Strategy")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No strategy-level results found.")

## 4. Sample Output Inspection

In [ ]:
# Show first 10 examples side-by-side
NUM_SHOW = 10

for i in range(NUM_SHOW):
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    for name, s in samples.items():
        if i < len(s):
            ex = s[i]
            if name == list(samples.keys())[0]:
                print(f"  Input:     {ex['input']}")
                print(f"  Strategy:  {ex['strategy']}")
                print(f"  Reference: {ex['reference']}")
            print(f"  {name:10s}: {ex['prediction']}")

## 5. Human Evaluation Template

Rate each sample on two dimensions (1-5 scale):
- **Meaning preservation**: Does the output convey the same underlying meaning as the sarcastic input?
- **Naturalness**: Does the output read like a natural, fluent headline?

Export samples below, then fill in ratings.

In [ ]:
# Generate human evaluation CSV template
eval_rows = []
best_model = list(samples.keys())[0] if samples else None

if best_model and samples[best_model]:
    for i, ex in enumerate(samples[best_model][:50]):
        row = {
            "id": i + 1,
            "input": ex["input"],
            "strategy": ex["strategy"],
            "reference": ex["reference"],
        }
        for name, s in samples.items():
            if i < len(s):
                row[f"pred_{name}"] = s[i]["prediction"]
                row[f"meaning_{name}"] = ""  # fill in 1-5
                row[f"naturalness_{name}"] = ""  # fill in 1-5
        eval_rows.append(row)

    df_eval = pd.DataFrame(eval_rows)
    eval_path = OUTPUTS_DIR / "human_eval_template.csv"
    eval_path.parent.mkdir(parents=True, exist_ok=True)
    df_eval.to_csv(eval_path, index=False)
    print(f"Human eval template saved to {eval_path}")
    display(df_eval.head())
else:
    print("No samples available yet. Run evaluation first.")